
# Database normalization with SMT solvers

In this notebook, we will
look at SMT solvers and see how they can be used to formalize and solve
certain problems. Particularly, we will show how we can encode functional dependencies
in Z3 to determine if a relation is normalized.


**Instructions:**
1. To get started, click on File on the top left and click "Save a copy in Drive."
This will give you an editable version of this document that you can use.
2. If you press `CMD`+`Enter` it runs the cell, and if you press `Shift`+`Enter` it runs the cell and goes to the next one.
3. Make sure you run all cells as you go through the notebook; some cells will not work properly unless the previous one
has been run too.
4. If you disconnect or are inactive for some time you should run all of the cells again.

## 0. Preliminaries (you should run this cell but there is no need to read it)

In [ ]:
!pip install z3-solver
!pip install git+https://github.com/crrivero/FormalMethodsTasting.git#subdirectory=core
from z3 import *
from tofmcore import showSolver
from math import log
from IPython.display import clear_output
clear_output()

## Encoding constraints in Z3

The goal of this notebook is to teach you about formal methods;
particularly, how you can use existing formal verification tools
(in this case, Z3) to analyze and solve your own problems.
Before we get started, let's look at some basic things we can do with Z3.

### Boolean

Suppose you have the following three boolean constraints and you want to check if there's a solution (an assignment of the variables) that satisfies all of them:

$$ x_1 \lor x_2 \lor x_3 $$

$$ \neg x_1 \implies \neg x_2$$

$$  x_1 \land x_3  $$

Let's see how we can do this using Z3.

In [ ]:
s = Solver() # initialize Z3 solver

# initialize variables

x1 = Bool('x_1') # declaring that x_1 is a boolean variable in Z3 which will be referred to as x1 in Python
x2 = Bool('x_2')
x3 = Bool('x_3')

# Note: we can also initialize multiple variables like so: x1, x2, x3 = Bools('x_1 x_2 x_3')

# we use s.add(.) to add a constraint to our solver s
# constraints can be made using many different operations such as Or, And, Not,
# equality, etc.

# here's how we would add the constraints above to our solver:

s.add( Or( x1, x2, x3 ) ) # add the first constraint
s.add( Implies( Not(x1), Not(x2) ) ) # add the second constraint
s.add( And( x1, x3 ) ) # add the third constraint

In [ ]:
# to view the constraints in our solver, we can use the following:
print( s )
# this prints the constraints as they appear in Z3 using Z3's notation

For better readability, this notebook also has a custom print function to view our constraints in LaTeX format, like so:

In [ ]:
showSolver( s )

In [ ]:
# we can use s.check() to run the solver and check whether its satisfiable:
print ( s.check() )

 "sat" means our system of constraints is satisfiable

In [ ]:
# after using s.check(),  we can use s.model() to output a solution if one exsits
solution = s.model()
print( solution )

Let's modify our system of constraint a bit and see if it's still satisfiable. Suppose we want to check if there's a solution where $x_1 = \neg x_3$. Let's see how we would do this with Z3.

In [ ]:
s.add( x1 == Not(x3) )
showSolver( s )

In [ ]:
print( s.check() ) # check if solution exists with new constraint

"unsat" means the system is not satisfiable, i.e., there is no assignment on the variables $x_1$, $x_2$, and $x_3$ that satisfies all the constraints we gave to the solver. **Note that if we were to run s.model() now we would get an error.**


## Database Normalization in Z3

In the previous notebook, we wrote some functions to help us determine if a relation satisfies 2NF. 
In this notebook, we will expand upon this to write a function to determine if a relation satisfies 3NF.

The relation we will test is below:

$$ R_1(\underline{enrollmentID}, studentId, studentName, courseID, courseName) $$

Just as before, the first thing we must do is determine the functional dependencies in the relation.



### Encoding Functional Dependencies

For two attributes X and Y, we say that X determines Y, $ X \rightarrow Y $, if for any two rows in the relation:

$$ row1.X = row2.X \rightarrow row1.Y = row2.Y $$

In other words, each value of X uniquely determines a value for Y.

**Complete the code below** to create a function that encodes a functional dependendency in Z3.


In [ ]:

# Create variables to represent two arbitrary rows in the relation
row1 = {
    'enrollmentID' : String('A.enrollmentID'),
    'studentID' : String('A.studentID'),
    'studentName' : String('A.studentName'),
    'courseID' : String('A.courseID'),
    'courseName' : String('A.courseName'),
}
row2 = {
    'enrollmentID' : String('B.enrollmentID'),
    'studentID' : String('B.studentID'),
    'studentName' : String('B.studentName'),
    'courseID' : String('B.courseID'),
    'courseName' : String('B.courseName'),
}

def functional_dependency(X, Y):
    return Implies( False ) # REPLACE THIS LINE    

# Initialize solver
s = Solver()

# Add two functional dependencies
s.add( functional_dependency('enrollmentID', 'studentID') )
s.add( functional_dependency('enrollmentID', 'courseName') )
    
showSolver(s)



Great! Next, to determine if a relation is normalized, we must check whether or not certain dependencies exist.

We can do this by adding the following constraint:

$$ row1.X = row2.X \land row1.Y \neq row2.Y $$

Here, you can see we are asking the solver to find a case where the two rows share the same value of X, but they do not have the same value for Y. 
This would mean there is a counterexample where X does not determine Y. If no such counterexample exists, then we have proven the functional dependency exists.

**Complete the code below** to create a function that tests whether or not a functional dependency exists.


In [ ]:

def has_fd(s, X, Y):
    s.push()

    s.add(
        And( False ) # REPLACE THIS LINE
    )

    result = s.check() == unsat # If we can't find a counterexample, then X -> Y

    # Remove the most recent constraint
    s.pop()

    return result

# Initialize solver
s = Solver()

# Add a functional dependency
s.add( functional_dependency('enrollmentID', 'studentID') )

# Check if the FD exists:
print("enrollmentID -> studentID :", has_fd(s, 'enrollmentID', 'studentID'))

# Check for a non-existent FD:
print("studentID -> courseName :", has_fd(s, 'studentID', 'courseName'))



### Determining 2NF

Now that we are able to represent functional dependencies, we can write a function to determine if a relation is in 3NF!

Recall that in order for a relation to satisfy 3NF it must first satisfy 2NF. 
In the previous notebook, we already wrote a function to do this, which has been provided here.

Let's take a look at the example again:

$$ R_1(\underline{enrollmentID}, studentId, studentName, courseID, courseName) $$

With the following functional dependencies:

$$ enrollmentID \rightarrow studentID $$
$$ enrollmentID \rightarrow studentName $$
$$ studentID \rightarrow studentName $$
$$ enrollmentID \rightarrow courseID $$
$$ enrollmentID \rightarrow courseName $$
$$ courseID \rightarrow courseName $$

**Complete the code below** to determine if the relation is in 2NF


In [ ]:

### THIS SHOULD BE IMPORTED SEPARATELY ###
def check_2nf(s, primary_key, non_prime_attributes):
    # Check that each non-prime attribute depends on the entire primary key
    for non_prime in non_prime_attributes:
        for prime in primary_key:
            dependency_exists = has_fd(s, prime, non_prime)
            if not dependency_exists:
                return False
    return True

# Create variables to represent two arbitrary rows in the relation
row1 = {
    'enrollmentID' : String('A.enrollmentID'),
    'studentID' : String('A.studentID'),
    'studentName' : String('A.studentName'),
    'courseID' : String('A.courseID'),
    'courseName' : String('A.courseName'),
}
row2 = {
    'enrollmentID' : String('B.enrollmentID'),
    'studentID' : String('B.studentID'),
    'studentName' : String('B.studentName'),
    'courseID' : String('B.courseID'),
    'courseName' : String('B.courseName'),
}

# Determine the primary key
primary_key = ['enrollmentID']
non_prime_attributes = ['studentID', 'studentName', 'courseID', 'courseName']

# Initialize solver
s = Solver()

# Create the functional dependencies:
s.add( False ) # REPLACE THESE LINES
s.add( False ) # REPLACE THESE LINES
s.add( False ) # REPLACE THESE LINES
s.add( False ) # REPLACE THESE LINES
s.add( False ) # REPLACE THESE LINES
s.add( False ) # REPLACE THESE LINES

print("Does the relation satisfy 2NF:", check_2nf(s, primary_key, non_prime_attributes))



### Determinine 3NF

Great! Now all that's left is to prove whether or not the relation satisfies 3NF.

Recall that a relation satisfies 3NF if there are no transitive dependencies. 
Transitive dependencies are function dependencies of the form:

$$ X \rightarrow Y $$
$$ Y \rightarrow Z $$

In other words, every non-prime attribute should depend only on the primary key. 
If any non-prime attribute depends on another non-prime attribute, the relation is not in 3NF.

**Complete the code below** to finish the function so that it returns true if the given relation satisfies 3NF.
Your function should return False for the example relation.


In [ ]:

def check_3nf(s, non_prime_attributes):
    # Check all pairs of non-prime attributes for transitive dependencies
    for attribute1 in non_prime_attributes:
        for attribute2 in non_prime_attributes:
            if True: # REPLACE THESE LINES
                return

print("Relation satisfies 3NF:", check_3nf(s, non_prime_attributes))



### Congratulations! You have now learned how to use Z3 to determine if a relation is normalized!


####If you'd like to continue your Z3 journey, you can start with this guide to learn more:
https://ericpony.github.io/z3py-tutorial/guide-examples.htm